# CRC Xenium: Compartment Annotation

Reproducible pipeline to assign spatial compartments (Tumor / Interface / Tissue)
to cells in the 10x Xenium P1 CRC dataset.

**Dataset:** [FFPE Human Colorectal Cancer with Immuno-Oncology Panel and Custom Add-on](https://www.10xgenomics.com/datasets/ffpe-human-colorectal-cancer-data-with-human-immuno-oncology-profiling-panel-and-custom-add-on-1-standard) (10x Genomics, Xenium V1). The tissue is an FFPE colorectal adenocarcinoma (Grade 2) profiled with 541 genes, including epithelial, immune, and stromal markers. Four morphology channels are available (DAPI, E-Cadherin, Vimentin, and a fourth stain).

**Goal:** Define spatial compartments around the tumor--tissue boundary so that ligand--receptor signaling can be benchmarked as a function of distance to the tumor. The pipeline assigns each cell to one of four compartments:

| Compartment | Definition |
|---|---|
| **Tumor** | Inside the smoothed tumor boundary |
| **50 micron** (Interface) | Within 50 µm outside the tumor boundary |
| **Tissue** | Beyond 50 µm outside the tumor boundary |
| **Excluded** | Outside the tissue mask or below the y = 4000 µm cutoff (glandular region) |

**Steps:**
1. Load raw Xenium cell data and expression matrix
2. Marker-based tumor vs non-tumor annotation
3. Build tissue mask from morphology images (E-Cadherin + Vimentin)
4. Build tumor region from cell-based tumor fraction
5. Assign compartments and compute signed distance to boundary
6. Validate with boundary overlay on morphology
7. Save `xenium_final.parquet`

In [ ]:
import numpy as np
import pandas as pd
import h5py
import tifffile
from scipy.sparse import csc_matrix
from scipy.ndimage import gaussian_filter, distance_transform_edt, binary_fill_holes
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
# ================================================================
# Paths — relative to the notebook directory
# ================================================================
DATA_DIR = 'data'            # symlink to raw Xenium outputs (Xenium_P1CRC/)
OUT_DIR = '.'                # output directory (same as notebook)

CELLS_PATH = f'{DATA_DIR}/cells.parquet'
H5_PATH = f'{DATA_DIR}/cell_feature_matrix.h5'
ECAD_PATH = f'{DATA_DIR}/morphology_focus/morphology_focus_0001.ome.tif'
VIM_PATH = f'{DATA_DIR}/morphology_focus/morphology_focus_0003.ome.tif'

# ================================================================
# Image parameters
# ================================================================
PIXEL_SIZE = 0.2125          # µm per pixel (Xenium instrument spec)

# ================================================================
# Spatial grid & downsampling
# ================================================================
GRID_SIZE = 10               # µm per grid pixel (shared resolution for cell grid & image grid)
IMG_DOWNSAMPLE = 47          # block-average factor: int(GRID_SIZE / PIXEL_SIZE) = 47
                             # effective pixel size after downsample: 47 * 0.2125 = 9.9875 µm ≈ GRID_SIZE

# ================================================================
# Tissue mask (from morphology images)
# ================================================================
TISSUE_SIGMA = 3             # Gaussian smoothing on downsampled image grids (in grid pixels = 30 µm)
TISSUE_PERCENTILE = 20       # threshold: percentile of non-zero smoothed intensity

# ================================================================
# Tumor region (from cell density)
# ================================================================
SIGMA = 8                    # Gaussian smoothing on tumor fraction grid (in grid pixels = 80 µm)
TUMOR_FRAC = 0.5             # tumor fraction threshold to define tumor region

# ================================================================
# Compartment assignment
# ================================================================
INTERFACE = 50               # µm buffer outside tumor boundary (defines '50 micron' compartment)
Y_CUTOFF = 4000              # µm; exclude glandular region below this y-coordinate

# ================================================================
# Visualization
# ================================================================
VIS_DOWNSAMPLE = 4           # stride-based downsample for boundary overlay plot

## 1. Load raw data

We load the unmodified Xenium outputs: `cells.parquet` (cell coordinates and QC metrics) and
`cell_feature_matrix.h5` (sparse gene-by-cell count matrix, 541 genes). The cell IDs in the
two files are in the same order, so no reindexing is needed, but we verify and align them
explicitly for safety.

In [ ]:
# Load cell metadata (raw Xenium output, no modifications)
cells = pd.read_parquet(CELLS_PATH)
print(f'Cells: {len(cells):,}')
print(f'Columns: {list(cells.columns)}')
print(f'X range: {cells.x_centroid.min():.0f} - {cells.x_centroid.max():.0f} µm')
print(f'Y range: {cells.y_centroid.min():.0f} - {cells.y_centroid.max():.0f} µm')

In [ ]:
# Load expression matrix
with h5py.File(H5_PATH, 'r') as f:
    matrix = f['matrix']
    gene_names = [g.decode() for g in matrix['features']['name'][:]]
    barcodes = [b.decode() for b in matrix['barcodes'][:]]
    data = matrix['data'][:]
    indices = matrix['indices'][:]
    indptr = matrix['indptr'][:]
    shape = matrix['shape'][:]

expr_sparse = csc_matrix((data, indices, indptr), shape=shape).T  # cells x genes
gene_to_idx = {g: i for i, g in enumerate(gene_names)}

# Match cell order to cells.parquet
barcode_to_idx = {b: i for i, b in enumerate(barcodes)}
cell_order = np.array([barcode_to_idx[cid] for cid in cells['cell_id'].values])
expr_sparse = expr_sparse[cell_order]

print(f'Expression: {expr_sparse.shape[0]:,} cells x {expr_sparse.shape[1]} genes')

def get_expr(gene):
    """Get expression vector for a gene."""
    return np.asarray(expr_sparse[:, gene_to_idx[gene]].todense()).flatten()

## 2. Marker-based tumor annotation

The dataset does not include pre-annotated cell types, so we identify tumor cells directly
from the Xenium expression counts using a marker-based gating strategy.

**Rationale:** CEACAM5 and CEACAM6 are highly expressed in colorectal carcinoma and serve as
the epithelial gate. Among epithelial cells, a *tumor score* (number of tumor-associated
markers detected) separates malignant from normal epithelium. Cells expressing normal
absorptive markers (FABP1, BEST4, etc.), stromal markers (FAP, ACTA2, etc.), or immune
markers (CD3D, CD68, etc.) are excluded to reduce false positives.

**Decision rules:**
- Epithelial gate: CEACAM5 > 0 OR CEACAM6 > 0
- Tumor score >= 2 (out of 8 markers)
- Not normal epithelial (normal score >= 2 AND tumor score < 3)
- Not stromal (stromal score >= 2)
- Not immune (immune score >= 2)

| Category | Markers |
|---|---|
| Epithelial gate | CEACAM5, CEACAM6 (any > 0) |
| Tumor score | CEACAM5, CEACAM6, MYC, MKI67, LAMC2, MMP7, LCN2, S100P (count > 0, need >= 2) |
| Normal epithelial | FABP1, BEST4, CLCA1, GUCA2A, AQP8, CA1, SLC26A3 |
| Stromal | FAP, ACTA2, PDGFRA, VWF, PECAM1 |
| Immune | CD3D, CD3E, CD8A, CD68, CD14, MS4A1, CD79A |

**Expected outcome:** ~31% of cells classified as tumor, concentrated in the upper portion
of the tissue section where adenocarcinoma is histologically evident.

In [ ]:
# Gene sets — CEACAM5/6 serve dual roles: epithelial gate AND tumor score
tumor_genes = ['CEACAM5', 'CEACAM6', 'MYC', 'MKI67', 'LAMC2', 'MMP7', 'LCN2', 'S100P']
normal_epi_genes = ['FABP1', 'BEST4', 'CLCA1', 'GUCA2A', 'AQP8', 'CA1', 'SLC26A3']
stromal_genes = ['FAP', 'ACTA2', 'PDGFRA', 'VWF', 'PECAM1']
immune_genes = ['CD3D', 'CD3E', 'CD8A', 'CD68', 'CD14', 'MS4A1', 'CD79A']

# Epithelial gate: CEACAM5 or CEACAM6 positive
ceacam5 = get_expr('CEACAM5')
ceacam6 = get_expr('CEACAM6')
is_epi = (ceacam5 > 0) | (ceacam6 > 0)

# Tumor score: count of tumor markers detected (> 0)
ts = np.zeros(len(cells))
for g in tumor_genes:
    ts += (get_expr(g) > 0).astype(float)

# Normal epithelial score
ns = np.zeros(len(cells))
for g in normal_epi_genes:
    ns += (get_expr(g) > 0).astype(float)

# Stromal score
ss = np.zeros(len(cells))
for g in stromal_genes:
    ss += (get_expr(g) > 0).astype(float)

# Immune score
ims = np.zeros(len(cells))
for g in immune_genes:
    ims += (get_expr(g) > 0).astype(float)

# Exclusion masks
is_normal_epi = is_epi & (ns >= 2) & (ts < 3)
is_stromal = ss >= 2
is_immune = ims >= 2

# Final tumor call
is_tumor = is_epi & (~is_normal_epi) & (~is_stromal) & (~is_immune) & (ts >= 2)

cells['is_tumor'] = is_tumor
print(f'Tumor:     {is_tumor.sum():>7,} ({is_tumor.mean()*100:.1f}%)')
print(f'Non-tumor: {(~is_tumor).sum():>7,} ({(~is_tumor).mean()*100:.1f}%)')

In [ ]:
# Visualize tumor vs non-tumor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x, y = cells['x_centroid'].values, cells['y_centroid'].values
t = is_tumor

ax = axes[0]
ax.scatter(x[~t], y[~t], s=0.1, c='#aaaaaa', alpha=0.3, rasterized=True)
ax.scatter(x[t], y[t], s=0.1, c='#c0392b', alpha=0.4, rasterized=True)
ax.set_aspect('equal'); ax.invert_yaxis()
ax.set_title(f'Tumor cells (red, n={t.sum():,})')
ax.set_xlabel('X (µm)'); ax.set_ylabel('Y (µm)')

ax = axes[1]
ax.scatter(x[t], y[t], s=0.1, c='#aaaaaa', alpha=0.3, rasterized=True)
ax.scatter(x[~t], y[~t], s=0.1, c='#2980b9', alpha=0.3, rasterized=True)
ax.set_aspect('equal'); ax.invert_yaxis()
ax.set_title(f'Non-tumor cells (blue, n={(~t).sum():,})')
ax.set_xlabel('X (µm)')

plt.tight_layout()
plt.show()

## 3. Tissue mask from morphology images

**Purpose:** Define the spatial extent of the tissue so that cells in empty slide regions
or imaging artifacts can be excluded.

**Rationale:** E-Cadherin (channel 0001) marks epithelial cells and Vimentin (channel 0003)
marks mesenchymal/stromal cells. Together, the two channels cover virtually all tissue
regions. We downsample the full-resolution images (0.2125 µm/pixel) to the same 10 µm
grid used for the cell density maps via block averaging (47x), smooth lightly, and threshold
at the 20th percentile of non-zero intensities. The union of the two masks, with holes
filled, defines the tissue region.

**Important:** No image transform (rotation, flip) is needed. The raw pixel coordinates,
scaled by the instrument pixel size (0.2125 µm), directly match the Xenium cell
centroid coordinates.

In [ ]:
# Load and downsample images to GRID_SIZE resolution
with tifffile.TiffFile(ECAD_PATH) as tif:
    ecad_full = tif.pages[0].asarray()
with tifffile.TiffFile(VIM_PATH) as tif:
    vim_full = tif.pages[0].asarray()

print(f'Full image shape: {ecad_full.shape}')
print(f'Full image extent: {ecad_full.shape[1]*PIXEL_SIZE:.0f} x {ecad_full.shape[0]*PIXEL_SIZE:.0f} µm')
print(f'Downsample factor: {IMG_DOWNSAMPLE}x (effective pixel: {IMG_DOWNSAMPLE * PIXEL_SIZE:.2f} µm)')

# Downsample by block averaging to GRID_SIZE resolution
def downsample_block(img, factor):
    """Downsample by averaging non-overlapping blocks."""
    h, w = img.shape
    nh, nw = h // factor, w // factor
    return img[:nh*factor, :nw*factor].reshape(nh, factor, nw, factor).mean(axis=(1, 3))

ecad = downsample_block(ecad_full, IMG_DOWNSAMPLE)
vim = downsample_block(vim_full, IMG_DOWNSAMPLE)
print(f'Downsampled grid shape: {ecad.shape}')

del ecad_full, vim_full

In [ ]:
# Build tissue mask from combined E-Cad + Vim signal
ecad_smooth = gaussian_filter(ecad, sigma=TISSUE_SIGMA)
vim_smooth = gaussian_filter(vim, sigma=TISSUE_SIGMA)

# Tissue = regions where either channel has signal above background
ecad_thresh = np.percentile(ecad_smooth[ecad_smooth > 0], TISSUE_PERCENTILE)
vim_thresh = np.percentile(vim_smooth[vim_smooth > 0], TISSUE_PERCENTILE)

tissue_mask_img = (ecad_smooth > ecad_thresh) | (vim_smooth > vim_thresh)
tissue_mask_img = binary_fill_holes(tissue_mask_img)

print(f'E-Cad threshold: {ecad_thresh:.1f} (p{TISSUE_PERCENTILE} of non-zero)')
print(f'Vim threshold:   {vim_thresh:.1f} (p{TISSUE_PERCENTILE} of non-zero)')
print(f'Tissue mask covers {tissue_mask_img.mean()*100:.1f}% of image grid')

## 4. Build tumor region from cell density

**Purpose:** Convert the per-cell tumor labels into a smooth, contiguous spatial region
that defines the tumor boundary for compartment assignment.

**Rationale:** Directly using per-cell `is_tumor` labels would produce a noisy, fragmented
boundary. Instead, we rasterize tumor and total cell counts onto a 10 µm grid, smooth
with a Gaussian kernel (sigma = 80 µm), and threshold the local tumor fraction at 0.5.
Holes inside the tumor region are filled. This yields a clean boundary that follows the
spatial distribution of tumor cells without being sensitive to individual misclassified cells.

In [ ]:
# Rasterize cells onto spatial grid — only cells above Y_CUTOFF contribute
# to the tumor fraction map, so that normal glands below the cutoff do not
# create spurious tumor boundaries
gx = (x / GRID_SIZE).astype(int)
gy = (y / GRID_SIZE).astype(int)
grid_shape = (gy.max() + 2, gx.max() + 2)

above_cutoff = y < Y_CUTOFF

tumor_grid = np.zeros(grid_shape)
np.add.at(tumor_grid, (gy[is_tumor & above_cutoff], gx[is_tumor & above_cutoff]), 1)

total_grid = np.zeros(grid_shape)
np.add.at(total_grid, (gy[above_cutoff], gx[above_cutoff]), 1)

# Smoothed tumor fraction
tumor_smooth = gaussian_filter(tumor_grid, sigma=SIGMA)
total_smooth = gaussian_filter(total_grid, sigma=SIGMA)
tumor_frac = np.divide(tumor_smooth, total_smooth,
                       where=total_smooth > 0,
                       out=np.zeros_like(tumor_smooth))

# Tumor region: high tumor fraction, filled holes
tumor_mask = binary_fill_holes(tumor_frac > TUMOR_FRAC)

print(f'Cell grid shape: {grid_shape}')
print(f'Cells above y={Y_CUTOFF} µm used for tumor map: {above_cutoff.sum():,}')
print(f'Tumor smoothing: sigma={SIGMA} grid pixels = {SIGMA * GRID_SIZE} µm')
print(f'Tumor fraction threshold: {TUMOR_FRAC}')
print(f'Tumor region covers {tumor_mask.mean()*100:.1f}% of cell grid')

## 5. Assign compartments

**Purpose:** Assign each cell to a spatial compartment based on its signed distance to the
tumor boundary, enabling downstream analysis of ligand–receptor signaling as a function of
position relative to the tumor.

**Method:** We compute an Euclidean Distance Transform (EDT) on the tumor region mask to get
a smooth, grid-based signed distance field (negative inside tumor, positive outside). Each
cell's distance is looked up from the grid at its (x, y) position.

**y = 4000 µm cutoff:** The lower portion of the tissue section (y > 4000 µm) contains
normal colonic glands that are CEACAM5/6-positive but non-malignant. Because these glands
express the same epithelial markers used in our tumor score, they produce scattered patches
of high tumor fraction that create spurious "tumor boundaries" far from the actual
adenocarcinoma. Excluding this region restricts the analysis to the contiguous tumor mass
in the upper portion of the section, where the tumor–stroma boundary is well-defined and
the distance-based compartment model is meaningful.

**Compartment assignment:**
- **Tumor:** signed distance ≤ 0 (inside the smoothed tumor boundary)
- **50 micron:** 0 < signed distance ≤ 50 µm (immediate peri-tumoral interface)
- **Tissue:** signed distance > 50 µm (bulk tissue beyond the interface)
- **Excluded:** outside tissue mask or y ≥ 4000 µm

We also assign finer-grained *tumor zones* (Tumor_core, Tumor_intermediate, Tumor_edge,
Interface_50um, Peri_tumor, Tissue_distal) for optional higher-resolution analyses.

In [ ]:
# Signed distance from tumor boundary (in µm)
# Negative = inside tumor, positive = outside tumor
dist_inside = distance_transform_edt(tumor_mask) * GRID_SIZE
dist_outside = distance_transform_edt(~tumor_mask) * GRID_SIZE
signed_dist_grid = dist_outside - dist_inside

# Per-cell signed distance
cell_dist = signed_dist_grid[gy, gx]

# Tissue mask: resample image-based mask to cell grid size
# The image grid and cell grid may differ in shape; use the cell grid coordinates
# to look up whether each cell falls in the tissue mask
img_gy = (y / GRID_SIZE).astype(int)
img_gx = (x / GRID_SIZE).astype(int)

# Clip to image mask bounds
img_gy_clipped = np.clip(img_gy, 0, tissue_mask_img.shape[0] - 1)
img_gx_clipped = np.clip(img_gx, 0, tissue_mask_img.shape[1] - 1)
in_tissue = tissue_mask_img[img_gy_clipped, img_gx_clipped]

# Assign compartments
compartment = np.full(len(cells), 'Excluded', dtype=object)
in_study = in_tissue & (y < Y_CUTOFF)

# Within study region: assign by distance to tumor boundary
compartment[in_study & (cell_dist <= 0)] = 'Tumor'                     # inside tumor
compartment[in_study & (cell_dist > 0) & (cell_dist <= INTERFACE)] = '50 micron'  # interface
compartment[in_study & (cell_dist > INTERFACE)] = 'Tissue'             # tissue

cells['compartment'] = compartment
cells['compartment_dist'] = cell_dist

# Tumor zones (finer resolution)
ZONE_THRESHOLDS = {
    'Tumor_core':         (-np.inf, -100),
    'Tumor_intermediate': (-100, -50),
    'Tumor_edge':         (-50, 0),
    'Interface_50um':     (0, 50),
    'Peri_tumor':         (50, 150),
    'Tissue_distal':      (150, np.inf),
}

tumor_zone = np.full(len(cells), 'Excluded', dtype=object)
for zone, (lo, hi) in ZONE_THRESHOLDS.items():
    mask = in_study & (cell_dist > lo) & (cell_dist <= hi)
    tumor_zone[mask] = zone

cells['tumor_zone'] = tumor_zone

# Summary
print('Compartment counts:')
for c in ['Tumor', '50 micron', 'Tissue', 'Excluded']:
    n = (compartment == c).sum()
    print(f'  {c:<12}: {n:>7,}')

print(f'\nTotal in study: {in_study.sum():,}')
print(f'Excluded:       {(~in_study).sum():,}')

In [ ]:
# Visualize compartments
fig, ax = plt.subplots(figsize=(10, 6))

colors = {
    'Excluded': '#e0e0e0',
    'Tissue':   '#a0b8d4',
    '50 micron':'#d4c090',
    'Tumor':    '#d4a0a0',
}

for c in ['Excluded', 'Tissue', '50 micron', 'Tumor']:
    m = compartment == c
    ax.scatter(x[m], y[m], s=0.1, c=colors[c], alpha=0.3, rasterized=True,
              label=f'{c} ({m.sum():,})')

ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_title('Compartment Annotation')
ax.legend(markerscale=20, fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

## 6. Boundary overlay on morphology

**Purpose:** Visually validate that the computed tumor boundary and interface contours align
with the histological features visible in the morphology images.

**Method:** We create an RGB composite of E-Cadherin (green, epithelial) and Vimentin
(magenta, mesenchymal) at 4x-downsampled resolution, then overlay the tumor boundary (red
solid line) and 50 µm interface (orange dashed line) as contours extracted from the
respective binary masks. The y = 4000 µm exclusion cutoff is also shown.

**Expected outcome:** The red tumor contour should trace the boundary of the E-Cadherin-bright
adenocarcinoma region in the upper portion of the tissue, while the orange interface contour
runs parallel ~50 µm outside it.

In [ ]:
from skimage.measure import find_contours

# Reload images at moderate downsample for visualization
with tifffile.TiffFile(ECAD_PATH) as tif:
    ecad_vis = tif.pages[0].asarray()[::VIS_DOWNSAMPLE, ::VIS_DOWNSAMPLE]
with tifffile.TiffFile(VIM_PATH) as tif:
    vim_vis = tif.pages[0].asarray()[::VIS_DOWNSAMPLE, ::VIS_DOWNSAMPLE]

# No transform needed — raw pixel*pixel_size matches cell coordinates
vis_pixel = PIXEL_SIZE * VIS_DOWNSAMPLE
extent = [0, ecad_vis.shape[1] * vis_pixel, ecad_vis.shape[0] * vis_pixel, 0]

# RGB composite
ecad_norm = np.clip(ecad_vis / np.percentile(ecad_vis, 99.5), 0, 1)
vim_norm = np.clip(vim_vis / np.percentile(vim_vis, 99.5), 0, 1)
rgb = np.zeros((*ecad_vis.shape, 3))
rgb[:, :, 1] = ecad_norm   # green = E-Cadherin
rgb[:, :, 0] = vim_norm    # red   = Vimentin
rgb[:, :, 2] = vim_norm    # blue  = Vimentin (magenta)

# Contours from the tumor and interface masks
tumor_contours = find_contours(tumor_mask.astype(float), 0.5)

dist_outside_grid = distance_transform_edt(~tumor_mask) * GRID_SIZE
interface_mask = dist_outside_grid <= INTERFACE
interface_contours = find_contours(interface_mask.astype(float), 0.5)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb, extent=extent)

for c in tumor_contours:
    ax.plot(c[:, 1] * GRID_SIZE, c[:, 0] * GRID_SIZE, 'r-', lw=1.5, alpha=0.9)
for c in interface_contours:
    ax.plot(c[:, 1] * GRID_SIZE, c[:, 0] * GRID_SIZE, 'orange', lw=1, ls='--', alpha=0.7)

ax.axhline(Y_CUTOFF, color='white', ls='--', lw=1, alpha=0.7)
ax.text(100, Y_CUTOFF + 80, f'y = {Y_CUTOFF} µm cutoff', color='white', fontsize=9)

ax.set_xlim(0, 6700)
ax.set_ylim(7300, 0)
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_title('Boundary overlay on E-Cadherin (green) + Vimentin (magenta)')
ax.legend(handles=[
    Line2D([0],[0], color='r', lw=2, label='Tumor boundary'),
    Line2D([0],[0], color='orange', lw=1.5, ls='--', label='50 µm interface'),
], loc='lower right', fontsize=10, facecolor='black', edgecolor='white', labelcolor='white')

plt.tight_layout()
plt.show()

## 7. Save annotated cell table

The final output is `xenium_final.parquet`, which augments the original Xenium `cells.parquet`
with three new columns:

| Column | Description |
|---|---|
| `is_tumor` | Boolean tumor annotation from marker gating |
| `compartment` | Spatial compartment (Tumor / 50 micron / Tissue / Excluded) |
| `compartment_dist` | Signed EDT distance to tumor boundary (µm; negative = inside tumor) |
| `tumor_zone` | Finer 6-level zone labels for optional analyses |

This file is used by all downstream scripts (benchmark, Figure 5 panels).

In [ ]:
out_path = f'{OUT_DIR}/xenium_final.parquet'
cells.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {cells.shape}')
print(f'Columns: {list(cells.columns)}')